# Naive Bayes Baseline — 3-Class Causal Relation Extraction

**Classes:**
- `Cause-Effect(e1,e2)` — entity 1 causes entity 2
- `Cause-Effect(e2,e1)` — entity 2 causes entity 1
- `Other` — all remaining relations (non-causal)

**Dataset:** SemEval-2010 Task 8 (Hendrickx et al., 2010)

**Features:** 2-word local context window around each entity (as in the original paper's NB baseline).

**Primary evaluation metric:** Macro F1 over the 2 Cause-Effect classes only (excluding Other).

In [1]:
# Cell 1: Setup
# =============
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..'))

TRAIN_PATH     = '../data/semeval2010/raw/TRAIN_FILE.TXT'
TEST_FULL_PATH = '../data/semeval2010/raw/TEST_FILE_FULL.TXT'

RESULTS_DIR    = '../results/naive_bayes/semeval2010/'

from src.shared.data_loader import (
    load_semeval_train,
    load_semeval_test_with_labels,
    get_label_distribution
)
from src.shared.evaluation import evaluate

from src.naive_bayes.features import extract_local_context, create_vectorizer
from src.naive_bayes.naive_bayes_model import build_naive_bayes_pipeline

print('Setup complete.')

Setup complete.


In [2]:
# Cell 2: Data Exploration — 3-Class View
# ========================================
# Load training data with 3-class mapping.
# All 17 non-Cause-Effect SemEval relations collapse into 'Other'.

train_examples = load_semeval_train(TRAIN_PATH, label_mode='3class')
print(f'Training examples: {len(train_examples)}')

print('\n=== CLASS DISTRIBUTION (training) ===')
for label, count, pct in get_label_distribution(train_examples):
    print(f'  {label:<26}  {count:>5}  ({pct:.1f}%)')

Training examples: 8000

=== CLASS DISTRIBUTION (training) ===
  Other                        6997  (87.5%)
  Cause-Effect(e2,e1)           659  (8.2%)
  Cause-Effect(e1,e2)           344  (4.3%)


In [3]:
# Cell 3: Feature Extraction + Train
# ====================================
# Features: 2-word local context window around each entity (same as
# the paper's NB baseline). Bag-of-words with unigrams only.

train_features = [extract_local_context(ex['sentence'], window=2)
                  for ex in train_examples]
train_labels   = [ex['label'] for ex in train_examples]

print(f'Feature string example: "{train_features[0]}"')
print(f'Label: {train_labels[0]}')

# Build pipeline: CountVectorizer (unigrams, lowercase) + MultinomialNB (alpha=1.0)
vectorizer = create_vectorizer()
model = build_naive_bayes_pipeline(vectorizer)

print('\nTraining...')
model.fit(train_features, train_labels)
print('Done.')
print(f"Vocabulary size: {len(model.named_steps['bow'].vocabulary_)} tokens")

Feature string example: "e1_l:an e1_l:arrayed e1:configuration e1_r:of e1_r:antenna e2_l:of e2_l:antenna e2:elements e2_r:."
Label: Other

Training...
Done.
Vocabulary size: 20186 tokens


In [4]:
# Cell 4: Predict + Evaluate
# ===========================
# Use TEST_FILE_FULL.TXT (which has ground-truth labels).
# Labels are mapped to 3-class before evaluation.

test_examples = load_semeval_test_with_labels(TEST_FULL_PATH, label_mode='3class')
print(f'Test examples: {len(test_examples)}')

print('\n=== CLASS DISTRIBUTION (test) ===')
for label, count, pct in get_label_distribution(test_examples):
    print(f'  {label:<26}  {count:>5}  ({pct:.1f}%)')

# Feature extraction
test_features = [extract_local_context(ex['sentence'], window=2)
                 for ex in test_examples]
y_true = [ex['label'] for ex in test_examples]
y_pred = model.predict(test_features).tolist()

# Save raw predictions
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, 'predictions.txt'), 'w', encoding='utf-8') as f:
    for label in y_pred:
        f.write(label + '\n')

# Evaluate with shared framework
# Results are printed AND saved to RESULTS_DIR
metrics = evaluate(
    y_true       = y_true,
    y_pred       = y_pred,
    model_name   = 'naive_bayes',
    dataset_name = 'semeval2010',
    output_dir   = RESULTS_DIR,
)

Test examples: 2717

=== CLASS DISTRIBUTION (test) ===
  Other                        2389  (87.9%)
  Cause-Effect(e2,e1)           194  (7.1%)
  Cause-Effect(e1,e2)           134  (4.9%)

  EVALUATION: naive_bayes  |  dataset: semeval2010

  *** PRIMARY METRIC ***
  Macro F1 (Cause-Effect classes only): 0.5876  (58.76%)

  Per-class results:
  Label                            P       R      F1   Support
  --------------------------------------------------------
  Cause-Effect(e1,e2)         1.0000  0.3657  0.5355       134
  Cause-Effect(e2,e1)         0.8047  0.5309  0.6398       194
  Other                       0.9343  0.9933  0.9629      2389

  Overall:
    Macro F1 (all 3 classes): 0.7127
    Micro F1:                 0.9293
    Accuracy:                 0.9293


Results saved to: ../results/naive_bayes/semeval2010/
  report.txt
  metrics.json
  confusion_matrix.png
